In [7]:
import random
import copy

In [ ]:
class Card:
    def __init__(self, color, value):
      self.color = color
      self.value = value
    def __str__(self):
      return f"{self.color} {self.value}"
    def __eq__(self, other):
      return self.color==other.color and self.value==other.value


def deck_generator():
  colors = ['Red', 'Blue', 'Green', 'Yellow']
  deck = []

  for color in colors:
    for number in range(10):       
        deck.append(Card(color,number)) #real uno deck has 2 of each 
        deck.append(Card(color,number))
    deck.append(Card(color,'Skip'))      
    deck.append(Card(color,'Skip'))      

  random.shuffle(deck)
  return deck


deck=deck_generator()
print(f"Deck size: {len(deck)}")

for card in deck[:5]: #seeing a sample
  print(card)


Deck size: 88
Red 3
Yellow 5
Blue 3
Blue 9
Green 4


In [ ]:
def initial_game_state(deck):
    state={
        'player1_cards':[deck.pop() for _ in range(5)], 
        'player2_cards':[deck.pop() for _ in range(5)], 
        'player3_cards':[deck.pop() for _ in range(5)], 
        'top_card':deck.pop(),
        'deck': deck,
        
        'winner': None,
        'current_player':1,
        'skip':False

    }

    return state

def get_valid_moves(hand, top_card):
    valid_card=[]

    for card in hand:

        if card.color==top_card.color or card.value==top_card.value: #typical uno rule
            valid_card.append(card)

    if not valid_card:
        return ['Draw']
    
    return valid_card

def apply_move(state, move, player_num):
    new_state=copy.deepcopy(state)
    user_display=f'player{player_num}_cards'

    if move=='Draw':
        if new_state['deck']:
            drawn = new_state['deck'].pop()
            new_state[user_display].append(drawn)
            print(f"  Player {player_num} draws: {drawn}")
        else:
            print(" The deck is empty! You cannot draw.")
    else:

        new_state[user_display].remove(move)
        new_state['top_card']=move

        if move.value=='Skip':
            new_state['skip']=True

    if len(new_state[user_display])==0:
        new_state['winner']=player_num

    return new_state


    


In [ ]:
def count_skips(hand):
    count=0
    for card in hand:
        if card.value=='Skip':
            count+=1
    return count


def defensive_evaluation(state, player_num):

    user_display=f'player{player_num}_cards'
    my_cards=len(state[user_display])

    opp_cards=0

    for p in [1, 2, 3]:

        if p!=player_num:
            opp_cards+=len(state[f'player{p}_cards'])
    opp_cards=opp_cards/2 #taking avg of the opp cards 

    skips=count_skips(state[user_display]
                      )

    #using the given formaualr
    score=50-6*my_cards + 2 * opp_cards + 4 * skips
    return score



def offensive_evaluation(state, player_num):

    user_display= state[f'player{player_num}_cards']
    my_cards=len(user_display)

    opp_cards=0

    for p in [1,2,3]:

        if p!=player_num:
            opp_cards+=len(state[f'player{p}_cards'])
            
    opp_cards= opp_cards/2


    skips=count_skips(user_display)
    score=50-6*my_cards + 2 * opp_cards + 4 * skips
    
    return score

In [9]:
def next_player(current):
    if current==3:
        return 1
    else:
        return current+1 #helps rotate bw the 3 players

In [ ]:
def minimax(state, depth, current_player, my_player):
    if depth==0 or state['winner']is not None: #if depth limit reached or winner found then stop
        return defensive_evaluation(state, my_player), None
    
    moves=get_valid_moves(state[f'player{current_player}_cards'], state['top_card'])
    #getting all valid moves


    if current_player==my_player:
        best_score=-9999
        best_move=None   

        for move in moves:
            #simulate
            new_state=apply_move(state, move, current_player)

            #move to next player
            next_p=next_player(current_player)
            
            #evaluate next moves
            score, _ =minimax(new_state, depth-1, next_p, my_player)

            #choosing the highest score move
            if score>best_score:
                best_score=score
                best_move=move         

        return best_score, best_move

    else: #opponent 

        best_score=9999

        for move in moves:
            new_state=apply_move(state, move, current_player)
            next_p=next_player(current_player)

            score, _ =minimax(new_state, depth-1, next_p, my_player)

            #minimize the score 
            if score<best_score:
                best_score=score

        return best_score, None
    


def p1_minimax(state):
    score, move=minimax(state, 3, current_player=1, my_player=1)
    print("Player 1 chooses:", move, " and score:", score)
    return move

In [ ]:
def expectimax(state, depth, current_player, my_player):

    if depth==0 or state['winner'] is not None:
        return offensive_evaluation(state, my_player), None

    moves=get_valid_moves(state[f'player{current_player}_cards'],state['top_card'])
    next_p=next_player(current_player)

    if current_player==my_player: #my turn 
        best_score=-9999
        best_move=None

        for move in moves:

            if move=='Draw':
                # chance simulating all possibilites 
                deck=state['deck']

                if not deck:
                    score=offensive_evaluation(state, my_player)

                else:

                    total=0

                    for card in deck:

                        new_state = copy.deepcopy(state)
                        new_state[f'player{my_player}_cards'].append(card)
                        new_state['deck'].remove(card)

                        s, _ = expectimax(new_state, depth-1, next_p, my_player)
                        total += s

                    score=total/len(deck) #each car has equal probability so

            else:
                new_state=apply_move(state, move, current_player)
                score, _ =expectimax(new_state, depth-1, next_p, my_player)

            if score>best_score:
                best_score=score
                best_move=move

        return best_score, best_move


    else: #opponent turn random
        if not moves:
            return offensive_evaluation(state, my_player), None

        move = random.choice(moves)
        new_state = apply_move(state, move, current_player)

        return expectimax(new_state, depth - 1, next_player(current_player), my_player)
    

def p2_expectimax_decision(state):

    score, move=expectimax(state, 3, current_player=2, my_player=2)

    print("Player 2 chooses:", move, "and score:", round(score, 2))
    return move